# 09 — Predict Rcnn / 卷积神经网络

**Chapter 26 — File 9 of 9 / 第26章 — 第9个文件（共9个）**

---

## Summary / 总结

This script demonstrates **detect kangaroos in photos with mask rcnn model**.

本脚本演示 **detect kangaroos in photos with mask rcnn model**。

---
## Step 1 — detect kangaroos in photos with mask rcnn model

In [ ]:
from os import listdir
from xml.etree import ElementTree
from numpy import zeros
from numpy import asarray
from numpy import expand_dims
from matplotlib import pyplot
from matplotlib.patches import Rectangle
from mrcnn.config import Config
from mrcnn.model import MaskRCNN
from mrcnn.model import mold_image
from mrcnn.utils import Dataset

---
## Step 2 — class that defines and loads the kangaroo dataset

In [ ]:
class KangarooDataset(Dataset):

---
## Step 3 — load the dataset definitions

In [ ]:
def load_dataset(self, dataset_dir, is_train=True):

---
## Step 4 — define one class

In [ ]:
self.add_class("dataset", 1, "kangaroo")

---
## Step 5 — define data locations

In [ ]:
images_dir = dataset_dir + '/images/'
		annotations_dir = dataset_dir + '/annots/'

---
## Step 6 — find all images

In [ ]:
for filename in listdir(images_dir):

---
## Step 7 — extract image id

In [ ]:
image_id = filename[:-4]

---
## Step 8 — skip bad images

In [ ]:
if image_id in ['00090']:
				continue

---
## Step 9 — skip all images after 150 if we are building the train set

In [ ]:
if is_train and int(image_id) >= 150:
				continue

---
## Step 10 — skip all images before 150 if we are building the test/val set

In [ ]:
if not is_train and int(image_id) < 150:
				continue
			img_path = images_dir + filename
			ann_path = annotations_dir + image_id + '.xml'

---
## Step 11 — add to dataset

In [ ]:
self.add_image('dataset', image_id=image_id, path=img_path, annotation=ann_path)

---
## Step 12 — load all bounding boxes for an image

In [ ]:
def extract_boxes(self, filename):

---
## Step 13 — load and parse the file

In [ ]:
root = ElementTree.parse(filename)
		boxes = list()

---
## Step 14 — extract each bounding box

In [ ]:
for box in root.findall('.//bndbox'):
			xmin = int(box.find('xmin').text)
			ymin = int(box.find('ymin').text)
			xmax = int(box.find('xmax').text)
			ymax = int(box.find('ymax').text)
			coors = [xmin, ymin, xmax, ymax]
			boxes.append(coors)

---
## Step 15 — extract image dimensions

In [ ]:
width = int(root.find('.//size/width').text)
		height = int(root.find('.//size/height').text)
		return boxes, width, height

---
## Step 16 — load the masks for an image

In [ ]:
def load_mask(self, image_id):

---
## Step 17 — get details of image

In [ ]:
info = self.image_info[image_id]

---
## Step 18 — define box file location

In [ ]:
path = info['annotation']

---
## Step 19 — load XML

In [ ]:
boxes, w, h = self.extract_boxes(path)

---
## Step 20 — create one array for all masks, each on a different channel

In [ ]:
masks = zeros([h, w, len(boxes)], dtype='uint8')

---
## Step 21 — create masks

In [ ]:
class_ids = list()
		for i in range(len(boxes)):
			box = boxes[i]
			row_s, row_e = box[1], box[3]
			col_s, col_e = box[0], box[2]
			masks[row_s:row_e, col_s:col_e, i] = 1
			class_ids.append(self.class_names.index('kangaroo'))
		return masks, asarray(class_ids, dtype='int32')

---
## Step 22 — load an image reference

In [ ]:
def image_reference(self, image_id):
		info = self.image_info[image_id]
		return info['path']

---
## Step 23 — define the prediction configuration

In [ ]:
class PredictionConfig(Config):

---
## Step 24 — define the name of the configuration

In [ ]:
NAME = "kangaroo_cfg"

---
## Step 25 — number of classes (background + kangaroo)

In [ ]:
NUM_CLASSES = 1 + 1

---
## Step 26 — simplify GPU config

In [ ]:
GPU_COUNT = 1
	IMAGES_PER_GPU = 1

---
## Step 27 — plot a number of photos with ground truth and predictions

In [ ]:
def plot_actual_vs_predicted(dataset, model, cfg, n_images=5):

---
## Step 28 — load image and mask

In [ ]:
for i in range(n_images):

---
## Step 29 — load the image and mask

In [ ]:
image = dataset.load_image(i)
		mask, _ = dataset.load_mask(i)

---
## Step 30 — convert pixel values (e.g. center)

In [ ]:
scaled_image = mold_image(image, cfg)

---
## Step 31 — convert image into one sample

In [ ]:
sample = expand_dims(scaled_image, 0)

---
## Step 32 — make prediction

In [ ]:
yhat = model.detect(sample, verbose=0)[0]

---
## Step 33 — define subplot

In [ ]:
pyplot.subplot(n_images, 2, i*2+1)

---
## Step 34 — turn off axis labels

In [ ]:
pyplot.axis('off')

---
## Step 35 — plot raw pixel data

In [ ]:
pyplot.imshow(image)
		if i == 0:
			pyplot.title('Actual')

---
## Step 36 — plot masks

In [ ]:
for j in range(mask.shape[2]):
			pyplot.imshow(mask[:, :, j], cmap='gray', alpha=0.3)

---
## Step 37 — get the context for drawing boxes

In [ ]:
pyplot.subplot(n_images, 2, i*2+2)

---
## Step 38 — turn off axis labels

In [ ]:
pyplot.axis('off')

---
## Step 39 — plot raw pixel data

In [ ]:
pyplot.imshow(image)
		if i == 0:
			pyplot.title('Predicted')
		ax = pyplot.gca()

---
## Step 40 — plot each box

In [ ]:
for box in yhat['rois']:

---
## Step 41 — get coordinates

In [ ]:
y1, x1, y2, x2 = box

---
## Step 42 — calculate width and height of the box

In [ ]:
width, height = x2 - x1, y2 - y1

---
## Step 43 — create the shape

In [ ]:
rect = Rectangle((x1, y1), width, height, fill=False, color='red')

---
## Step 44 — draw the box

In [ ]:
ax.add_patch(rect)

---
## Step 45 — show the figure

In [ ]:
pyplot.show()

---
## Step 46 — load the train dataset

In [ ]:
train_set = KangarooDataset()
train_set.load_dataset('kangaroo', is_train=True)
train_set.prepare()
print('Train: %d' % len(train_set.image_ids))

---
## Step 47 — load the test dataset

In [ ]:
test_set = KangarooDataset()
test_set.load_dataset('kangaroo', is_train=False)
test_set.prepare()
print('Test: %d' % len(test_set.image_ids))

---
## Step 48 — create config

In [ ]:
cfg = PredictionConfig()

---
## Step 49 — define the model

In [ ]:
model = MaskRCNN(mode='inference', model_dir='./', config=cfg)

---
## Step 50 — load model weights

In [ ]:
model_path = 'mask_rcnn_kangaroo_cfg_0005.h5'
model.load_weights(model_path, by_name=True)

---
## Step 51 — plot predictions for train dataset

In [ ]:
plot_actual_vs_predicted(train_set, model, cfg)

---
## Step 52 — plot predictions for test dataset

In [ ]:
plot_actual_vs_predicted(test_set, model, cfg)

---
## Learning Notes / 学习笔记

- **概念**: detect kangaroos in photos with mask rcnn model 是机器学习中的常用技术。  
  *detect kangaroos in photos with mask rcnn model is a common technique in machine learning.*

- **ML 应用**: 本示例展示了如何在实践中应用该技术。  
  *This example shows how to apply the technique in practice.*

---
## Complete Code / 完整代码一览

Below is the full code for quick reference. / 以下是完整代码，供快速参考。

In [ ]:
# ===============================
# Predict Rcnn / 卷积神经网络
# Complete Code / 完整代码
# ===============================

# detect kangaroos in photos with mask rcnn model
from os import listdir
from xml.etree import ElementTree
from numpy import zeros
from numpy import asarray
from numpy import expand_dims
from matplotlib import pyplot
from matplotlib.patches import Rectangle
from mrcnn.config import Config
from mrcnn.model import MaskRCNN
from mrcnn.model import mold_image
from mrcnn.utils import Dataset

# class that defines and loads the kangaroo dataset
class KangarooDataset(Dataset):
	# load the dataset definitions
	def load_dataset(self, dataset_dir, is_train=True):
		# define one class
		self.add_class("dataset", 1, "kangaroo")
		# define data locations
		images_dir = dataset_dir + '/images/'
		annotations_dir = dataset_dir + '/annots/'
		# find all images
		for filename in listdir(images_dir):
			# extract image id
			image_id = filename[:-4]
			# skip bad images
			if image_id in ['00090']:
				continue
			# skip all images after 150 if we are building the train set
			if is_train and int(image_id) >= 150:
				continue
			# skip all images before 150 if we are building the test/val set
			if not is_train and int(image_id) < 150:
				continue
			img_path = images_dir + filename
			ann_path = annotations_dir + image_id + '.xml'
			# add to dataset
			self.add_image('dataset', image_id=image_id, path=img_path, annotation=ann_path)

	# load all bounding boxes for an image
	def extract_boxes(self, filename):
		# load and parse the file
		root = ElementTree.parse(filename)
		boxes = list()
		# extract each bounding box
		for box in root.findall('.//bndbox'):
			xmin = int(box.find('xmin').text)
			ymin = int(box.find('ymin').text)
			xmax = int(box.find('xmax').text)
			ymax = int(box.find('ymax').text)
			coors = [xmin, ymin, xmax, ymax]
			boxes.append(coors)
		# extract image dimensions
		width = int(root.find('.//size/width').text)
		height = int(root.find('.//size/height').text)
		return boxes, width, height

	# load the masks for an image
	def load_mask(self, image_id):
		# get details of image
		info = self.image_info[image_id]
		# define box file location
		path = info['annotation']
		# load XML
		boxes, w, h = self.extract_boxes(path)
		# create one array for all masks, each on a different channel
		masks = zeros([h, w, len(boxes)], dtype='uint8')
		# create masks
		class_ids = list()
		for i in range(len(boxes)):
			box = boxes[i]
			row_s, row_e = box[1], box[3]
			col_s, col_e = box[0], box[2]
			masks[row_s:row_e, col_s:col_e, i] = 1
			class_ids.append(self.class_names.index('kangaroo'))
		return masks, asarray(class_ids, dtype='int32')

	# load an image reference
	def image_reference(self, image_id):
		info = self.image_info[image_id]
		return info['path']

# define the prediction configuration
class PredictionConfig(Config):
	# define the name of the configuration
	NAME = "kangaroo_cfg"
	# number of classes (background + kangaroo)
	NUM_CLASSES = 1 + 1
	# simplify GPU config
	GPU_COUNT = 1
	IMAGES_PER_GPU = 1

# plot a number of photos with ground truth and predictions
def plot_actual_vs_predicted(dataset, model, cfg, n_images=5):
	# load image and mask
	for i in range(n_images):
		# load the image and mask
		image = dataset.load_image(i)
		mask, _ = dataset.load_mask(i)
		# convert pixel values (e.g. center)
		scaled_image = mold_image(image, cfg)
		# convert image into one sample
		sample = expand_dims(scaled_image, 0)
		# make prediction
		yhat = model.detect(sample, verbose=0)[0]
		# define subplot
		pyplot.subplot(n_images, 2, i*2+1)
		# turn off axis labels
		pyplot.axis('off')
		# plot raw pixel data
		pyplot.imshow(image)
		if i == 0:
			pyplot.title('Actual')
		# plot masks
		for j in range(mask.shape[2]):
			pyplot.imshow(mask[:, :, j], cmap='gray', alpha=0.3)
		# get the context for drawing boxes
		pyplot.subplot(n_images, 2, i*2+2)
		# turn off axis labels
		pyplot.axis('off')
		# plot raw pixel data
		pyplot.imshow(image)
		if i == 0:
			pyplot.title('Predicted')
		ax = pyplot.gca()
		# plot each box
		for box in yhat['rois']:
			# get coordinates
			y1, x1, y2, x2 = box
			# calculate width and height of the box
			width, height = x2 - x1, y2 - y1
			# create the shape
			rect = Rectangle((x1, y1), width, height, fill=False, color='red')
			# draw the box
			ax.add_patch(rect)
	# show the figure
	pyplot.show()

# load the train dataset
train_set = KangarooDataset()
train_set.load_dataset('kangaroo', is_train=True)
train_set.prepare()
print('Train: %d' % len(train_set.image_ids))
# load the test dataset
test_set = KangarooDataset()
test_set.load_dataset('kangaroo', is_train=False)
test_set.prepare()
print('Test: %d' % len(test_set.image_ids))
# create config
cfg = PredictionConfig()
# define the model
model = MaskRCNN(mode='inference', model_dir='./', config=cfg)
# load model weights
model_path = 'mask_rcnn_kangaroo_cfg_0005.h5'
model.load_weights(model_path, by_name=True)
# plot predictions for train dataset
plot_actual_vs_predicted(train_set, model, cfg)
# plot predictions for test dataset
plot_actual_vs_predicted(test_set, model, cfg)